In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
excel_path = "/home/ubuntu/giodir/digitalPathology/data/aiFlopp/Trento AIFLOPP.xlsx"
sheet_name = "Referti Trento definitivi"

In [3]:
# open excel sheet in pandas

df = pd.read_excel(excel_path, sheet_name=sheet_name, header=[0])

In [4]:
df.head()

,,CODICE CASO,CORE,Unnamed: 3,NUMERO FRUSTOLI,LETTORE,DIAGNOSI,DIAGNOSI DOPO IMMUNO,GLEASON Principale,GLEASON Secondario,...,GLEASON Secondario.1,% PATTERN 3.1,% PATTERN 4.1,% PATTERN 5.1,LUNGHEZZA CORE BIOPTICO MM,LUNGHEZZA TUMORE MM.1,% TUMORE.1,INVASIONE PERINEURALE.1,CRIBRIFORME.1,GG ISUP
0,25-I-10859,Tn001,1,NaN,2,MN,4,LEGENDA TABELLA,4.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SCANNED,Tn001,2,NaN,2,MN,4,DIAGNOSI,4.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Tn001,3,NaN,2,MN,4,LUNGHEZZA TUMORE MM,4.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Tn001,4,IBEX PAIGE,2,MN,4,INVASIONE PERINEURALE,4.0,5.0,...,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Tn001,5,IBEX PAIGE,2,MN,4,CRIBRIFORME,4.0,5.0,...,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.columns.to_list()

['   ',
 'CODICE CASO',
 'CORE',
 'Unnamed: 3',
 'NUMERO FRUSTOLI',
 'LETTORE',
 'DIAGNOSI',
 'DIAGNOSI DOPO IMMUNO',
 'GLEASON Principale',
 'GLEASON Secondario',
 '% PATTERN 3',
 '% PATTERN 4',
 '% PATTERN 5',
 'LUNGHEZZA CORE BIOPTICO',
 'LUNGHEZZA TUMORE MM',
 '% TUMORE',
 'INVASIONE PERINEURALE',
 'CRIBRIFORME',
 'GG ISUP PER SINGOLO CORE',
 'PIN DI ALTO GRADO',
 "QUALITA' VETRINO",
 'PIEGHE',
 'SPESSORE',
 'COLORAZIONE',
 'CONTAMINAZIONE',
 'BOLLE',
 'FRAMMENTAZIONE',
 'ARTEFATTI VARI DA PROCESSAZIONE',
 "QUALITA' TOTALE",
 'Unnamed: 29',
 "QUALITA' PERCEPITA",
 'TEMPO NECESSARIO PER LETTURA VETRINO (MIN)',
 "QUALITA' PER WSI - FUORI FUOCO",
 'SCANNERIZZAZIONE PARZIALE',
 'TEMPO NECESSARIO PER LETTURA WSI',
 'WSI',
 'DIAGNOSI.1',
 'LETTORE.1',
 'GLEASON Principale.1',
 'GLEASON Secondario.1',
 '% PATTERN 3.1',
 '% PATTERN 4.1',
 '% PATTERN 5.1',
 'LUNGHEZZA CORE BIOPTICO MM',
 'LUNGHEZZA TUMORE MM.1',
 '% TUMORE.1',
 'INVASIONE PERINEURALE.1',
 'CRIBRIFORME.1',
 'GG ISUP']

In [18]:
columns_to_keep = [
    'CODICE CASO',
    'CORE',
    'DIAGNOSI'
]

columns_new_names = {
    'CODICE CASO': 'patient_id',
    'CORE': 'bersaglio',
    'DIAGNOSI': 'diagnosi'
}


filtered_df = df[columns_to_keep]
filtered_df.columns = [columns_new_names.get(col, col) for col in filtered_df.columns]

In [19]:
filtered_df.head()

,patient_id,bersaglio,diagnosi
0,Tn001,1,4
1,Tn001,2,4
2,Tn001,3,4
3,Tn001,4,4
4,Tn001,5,4


In [20]:
# Remove rows with missing or not valid valies (valid val 0-5)
filtered_df = filtered_df[filtered_df['diagnosi'].notnull() & filtered_df["diagnosi"].isin([0, 1, 2, 3, 4, 5])]

In [21]:
filtered_df["diagnosi"].value_counts()

diagnosi
0    249
4    169
2     44
Name: count, dtype: int64

In [22]:
filtered_df['diagnosi_bool'] = np.where(filtered_df['diagnosi'].isin([1, 3, 4, 5]), 1, 0)

In [23]:
filtered_df.head()

,patient_id,bersaglio,diagnosi,diagnosi_bool
0,Tn001,1,4,1
1,Tn001,2,4,1
2,Tn001,3,4,1
3,Tn001,4,4,1
4,Tn001,5,4,1


In [24]:
# Parse labels to get the bag_id

parsed_case_code = filtered_df["patient_id"].apply(lambda x: str(int(str(x)[2:])))
filtered_df['tn_bag_id'] = "TN_" + parsed_case_code + "_" + filtered_df["bersaglio"].astype(str)

In [25]:
# Associate to each row the correct bag id looking at the folder

features_dir = Path("/home/ubuntu/giodir/digitalPathology/data/features/uni_features_TR")

def find_bag_id(tn_bag_id):
    for file in features_dir.glob(f"{tn_bag_id}_*.npz"):
        return file.stem  # return the filename without extension
    return None  # if no file is found

filtered_df['bag_id'] = filtered_df['tn_bag_id'].apply(find_bag_id)

print("not matched")
filtered_df[filtered_df["bag_id"].isnull()]

not matched


,patient_id,bersaglio,diagnosi,diagnosi_bool,tn_bag_id,bag_id
417,Tn036,,0,0,TN_36_,NaN


In [26]:
filtered_df

,patient_id,bersaglio,diagnosi,diagnosi_bool,tn_bag_id,bag_id
0,Tn001,1,4,1,TN_1_1,TN_1_1_113600
1,Tn001,2,4,1,TN_1_2,TN_1_2_113717
2,Tn001,3,4,1,TN_1_3,TN_1_3_113829
3,Tn001,4,4,1,TN_1_4,TN_1_4_114005
4,Tn001,5,4,1,TN_1_5,TN_1_5_114134
...,...,...,...,...,...,...
461,Tn040,8,0,0,TN_40_8,TN_40_8_174826
462,Tn040,9,0,0,TN_40_9,TN_40_9_174954
463,Tn040,10,0,0,TN_40_10,TN_40_10_175130
464,Tn040,11,0,0,TN_40_11,TN_40_11_175256


In [28]:
# Filter to keep only the analyzed cases

features_dir = Path("/home/ubuntu/giodir/digitalPathology/data/features/uni_features_TR")

available_bags = {path.stem for path in features_dir.glob("*.npz")}
print("Analyzed bags:", len(available_bags))

avail_df = filtered_df[filtered_df['bag_id'].isin(available_bags)]

print(f"Total cases: {len(filtered_df)}, Available cases: {len(avail_df)},")

Analyzed bags: 466
Total cases: 462, Available cases: 461,


In [29]:
out_df = avail_df[["bag_id", "diagnosi_bool"]]
out_df.rename(columns={"diagnosi_bool": "label"}, inplace=True)

outfile_file = Path("/home/ubuntu/giodir/digitalPathology/data/labels/tn_base_labels/tn_base_labels_tn.csv")
outfile_file.parent.mkdir(parents=True, exist_ok=True)

out_df.to_csv(outfile_file, index=False)